In [ ]:
# =============================================================================
# mc_block_2 — Dynamic Stability MC
# Block A: near-manifold (on, below, above)
# Block B: moderate off-manifold (moderate below, moderate above)
#
# Deliverables:
#   - Attractor classification (low / high / interior_fixed / cycle_like / wandering)
#   - Break diagnostics on g (break_score_mean, break_score_var, collapse_flag)
#   - Country-level attractor map (share_interior_fixed, share_cycle_like, collapse_rate)
#   - Timescale ratio (log10(chi/sigma)) as primary sweep axis
#
# Theory compliance:
#   - include_structural_drag=True  uses Γ_eff(ψ) = Γ - D(ψ) in g*.
#   - Exact positivity-preserving Riccati step for g (no RK4 for g).
#   - Linear interpolation for all grid lookups.
# =============================================================================

import os
import json
import platform
from datetime import datetime
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------
# Deterministic linear algebra
# -----------------------------
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


# =============================================================================
# CONFIG
# =============================================================================
MC_NAME = "mc_block_2"
RUN_TS = datetime.now().strftime("%y%m%d_%H%M")

BASE_OUT_DIR = "/content/drive/MyDrive/vsc_saves"
OUT_DIR = os.path.join(BASE_OUT_DIR, f"{MC_NAME}_{RUN_TS}")
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 20260212

MC_CONFIG = {
    # Countries
    "N_COUNTRIES": 150,
    "psi_grid_n": 200,
    "psi0_grid_n": 21,

    # Structural primitive ranges
    "Gamma_range": (0.01, 0.05),
    "rho_bar_range": (1.0, 5.0),
    "a_range": (0.5, 2.0),
    "c_lambda_range": (0.1, 1.0),
    "eta_lambda_range": (1.01, 2.0),
    "c_Delta_range": (0.1, 1.0),
    "eta_Delta_range": (1.0, 2.0),
    "c_delta_range": (0.05, 0.5),
    "eta_delta_range": (1.0, 2.0),
    "psi_L_range": (0.0, 0.3),
    "Wbar_range": (1.0, 50.0),
    "kappa_c_range": (0.05, 2.0),

    # Global validity / grid coverage filter
    "max_bad_share": 0.05,

    # Solvency tolerance
    "eps_solv": 1e-12,

    # g* tolerance (for "OK" checks)
    "tol_g": 1e-12,

    # Dynamics draw ranges
    "chi_range": (1e-1, 1e2),
    "sigma_range": (1e-3, 1e0),
    "omega_mult_range": (0.5, 2.0),

    # Ratio bins for summaries
    "ratio_bins_log10": [-3, -2, -1, 0, 1, 2, 3, 4, 5],

    # Derivative safety
    "DERIV_RHO_MIN": 1e-8,

    # Omega floor (prevents pathological tanh scaling)
    "OMEGA_MIN": 1e-8,
}

SIM_CONFIG = {
    # Dynamic horizon parameters
    "T_base": 100.0,
    "T_cap": 5000.0,
    "SIGMA_SCALE_REF": 0.05,

    "dt": 0.02,
    "eps_psi": 1e-4,
    "eps_g": 1e-4,
    "stable_window": 200,
}


# --- IC magnitudes ---
IC_NEAR_EPS = 0.05
IC_FAR_BELOW_MULT = 0.50
IC_FAR_ABOVE_MULT = 1.00
IC_FAR_ABOVE_CAP_MULT = 0.50


def g_scale(gstar):
    return 1.0 + abs(gstar)


# =============================================================================
# Utilities
# =============================================================================
def draw_uniform(rng, low, high):
    return rng.uniform(low, high)


def mono_nondec(x, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if not np.all(np.isfinite(x)):
        return False
    return bool(np.all(np.diff(x) >= -tol))


def mono_noninc(x, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if not np.all(np.isfinite(x)):
        return False
    return bool(np.all(np.diff(x) <= tol))


def save_df(df, tag):
    path = os.path.join(OUT_DIR, f"{MC_NAME}_{tag}_{RUN_TS}.csv")
    df.to_csv(path, index=False)
    return path


def save_plot_scatter_with_binned(df, x_col, y_col, bins_df, y_bin_col, title, out_stub):
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    ax.scatter(df[x_col], df[y_col], s=15, c="0.6", alpha=0.65, edgecolors="none")
    bb = bins_df.dropna(subset=["bin_center", y_bin_col])
    if not bb.empty:
        ax.plot(bb["bin_center"], bb[y_bin_col], color="0.2", linewidth=2.0)
    ax.set_title(title)
    ax.set_xlabel("log10(chi/sigma)")
    ax.set_ylabel(y_col)
    ax.grid(alpha=0.25, color="0.8")
    fig.tight_layout()
    png = os.path.join(OUT_DIR, f"{out_stub}_{RUN_TS}.png")
    pdf = os.path.join(OUT_DIR, f"{out_stub}_{RUN_TS}.pdf")
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    return png, pdf


# =============================================================================
# Micro primitives + objects
# =============================================================================
def draw_country_theta(rng, cfg):
    rho_bar = draw_uniform(rng, *cfg["rho_bar_range"])
    a = draw_uniform(rng, *cfg["a_range"])
    b = a / (2.0 * rho_bar)
    r0 = 0.0
    return {
        "rho_bar": rho_bar, "a": a, "b": b, "r0": r0,
        "c_lambda": draw_uniform(rng, *cfg["c_lambda_range"]),
        "eta_lambda": draw_uniform(rng, *cfg["eta_lambda_range"]),
        "c_Delta": draw_uniform(rng, *cfg["c_Delta_range"]),
        "eta_Delta": draw_uniform(rng, *cfg["eta_Delta_range"]),
        "c_delta": draw_uniform(rng, *cfg["c_delta_range"]),
        "eta_delta": draw_uniform(rng, *cfg["eta_delta_range"]),
        "psi_L": draw_uniform(rng, *cfg["psi_L_range"]),
        "Wbar": draw_uniform(rng, *cfg["Wbar_range"]),
        "Gamma": draw_uniform(rng, *cfg["Gamma_range"]),
        "kappa_c": draw_uniform(rng, *cfg["kappa_c_range"]),
    }


def r_func(rho, r0, a, b):
    return r0 + a * rho - b * rho**2


def r_prime(rho, a, b):
    return a - 2.0 * b * rho


def lam_func(rho, c_lam, eta_lam):
    return 1.0 - np.exp(-c_lam * rho**eta_lam)


def lam_prime(rho, c_lam, eta_lam):
    arr = np.asarray(rho, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    valid = np.isfinite(arr) & (arr >= 0.0)
    if np.any(valid):
        f = np.exp(-c_lam * arr[valid]**eta_lam)
        g = c_lam * eta_lam * arr[valid]**(eta_lam - 1.0)
        out[valid] = f * g
    return out if arr.ndim > 0 else float(out)


def lam_second(rho, c_lam, eta_lam, DERIV_RHO_MIN):
    arr = np.asarray(rho, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    valid = np.isfinite(arr) & (arr >= DERIV_RHO_MIN)
    if np.any(valid):
        f = np.exp(-c_lam * arr[valid]**eta_lam)
        g = c_lam * eta_lam * arr[valid]**(eta_lam - 1.0)
        gp = c_lam * eta_lam * (eta_lam - 1.0) * arr[valid]**(eta_lam - 2.0)
        out[valid] = f * (gp - g**2)
    return out if arr.ndim > 0 else float(out)


def Delta_func(rho, c_Delta, eta_Delta):
    return c_Delta * rho**eta_Delta


def Delta_prime(rho, c_Delta, eta_Delta):
    arr = np.asarray(rho, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    valid = np.isfinite(arr) & (arr >= 0.0)
    if np.any(valid):
        x = arr[valid]
        out[valid] = c_Delta * eta_Delta * np.power(x, eta_Delta - 1.0)
    return out if arr.ndim > 0 else float(out)


def Delta_second(rho, c_Delta, eta_Delta, DERIV_RHO_MIN):
    arr = np.asarray(rho, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    valid = np.isfinite(arr) & (arr >= DERIV_RHO_MIN)
    if np.any(valid):
        out[valid] = c_Delta * eta_Delta * (eta_Delta - 1.0) * arr[valid]**(eta_Delta - 2.0)
    return out if arr.ndim > 0 else float(out)


def delta_func(rho, c_delta, eta_delta):
    return c_delta * rho**eta_delta


def B_func(rho, c_lam, eta_lam, c_Delta, eta_Delta):
    lam_p = lam_prime(rho, c_lam, eta_lam)
    lam = lam_func(rho, c_lam, eta_lam)
    D = Delta_func(rho, c_Delta, eta_Delta)
    D_p = Delta_prime(rho, c_Delta, eta_Delta)
    return lam_p * D + lam * D_p


def B_prime(rho, c_lam, eta_lam, c_Delta, eta_Delta, DERIV_RHO_MIN):
    lam = lam_func(rho, c_lam, eta_lam)
    lam_p = lam_prime(rho, c_lam, eta_lam)
    lam_pp = lam_second(rho, c_lam, eta_lam, DERIV_RHO_MIN)
    D = Delta_func(rho, c_Delta, eta_Delta)
    D_p = Delta_prime(rho, c_Delta, eta_Delta)
    D_pp = Delta_second(rho, c_Delta, eta_Delta, DERIV_RHO_MIN)
    return lam_pp * D + 2.0 * lam_p * D_p + lam * D_pp


def solve_rho_star(psi, rho_bar, a, b, c_lam, eta_lam, c_Delta, eta_Delta,
                   tol=1e-8, max_iter=200):
    def F(rho):
        return r_prime(rho, a, b) - (1.0 - psi) * B_func(rho, c_lam, eta_lam, c_Delta, eta_Delta)

    rho_lo_int = 0.0
    rho_hi = rho_bar
    f_lo = F(rho_lo_int)
    f_hi = F(rho_hi)
    if not np.isfinite(f_hi):
        return np.nan, "nonfinite"
    if np.isfinite(f_lo) and abs(f_lo) < tol:
        return 0.0, "corner_low"
    if np.isfinite(f_hi) and abs(f_hi) < tol:
        return rho_hi, "corner_high"
    if np.isfinite(f_lo) and (f_lo * f_hi < 0.0):
        lo, hi = rho_lo_int, rho_hi
        for _ in range(max_iter):
            mid = 0.5 * (lo + hi)
            f_mid = F(mid)
            if not np.isfinite(f_mid):
                return np.nan, "nonfinite"
            if abs(f_mid) < tol or (hi - lo) < tol:
                return mid, "interior"
            if np.sign(f_mid) == np.sign(f_lo):
                lo, f_lo = mid, f_mid
            else:
                hi = mid
        return np.nan, "no_converge"
    if np.isfinite(f_lo) and (f_lo > 0.0) and (f_hi > 0.0):
        return rho_bar, "corner_high"
    if np.isfinite(f_lo) and (f_lo < 0.0) and (f_hi < 0.0):
        return 0.0, "corner_low"
    return np.nan, "no_bracket"


def g_star_stationary(psi, Gamma_eff, kappa_c):
    if (not np.isfinite(psi)) or (not np.isfinite(Gamma_eff)) or (not np.isfinite(kappa_c)):
        return np.nan
    if Gamma_eff <= 0.0:
        return 0.0
    if psi <= 0.0:
        return Gamma_eff
    disc = 1.0 + 4.0 * kappa_c * psi * Gamma_eff
    if disc <= 0.0:
        return 0.0
    return (-1.0 + np.sqrt(disc)) / (2.0 * kappa_c * psi)


def build_manifold_objects(theta, cfg):
    """
    Returns arrays on psi_grid, validity masks, and diagnostic flags.
    """
    DERIV_RHO_MIN = cfg["DERIV_RHO_MIN"]

    rho_bar, a, b, r0 = theta["rho_bar"], theta["a"], theta["b"], theta["r0"]
    c_lam, eta_lam = theta["c_lambda"], theta["eta_lambda"]
    c_Delta, eta_Delta = theta["c_Delta"], theta["eta_Delta"]
    c_delta, eta_delta = theta["c_delta"], theta["eta_delta"]
    psi_L, Wbar, Gamma, kappa_c = theta["psi_L"], theta["Wbar"], theta["Gamma"], theta["kappa_c"]

    psi_grid = np.linspace(psi_L, 1.0, cfg["psi_grid_n"])

    rho0, st0 = solve_rho_star(0.0, rho_bar, a, b, c_lam, eta_lam, c_Delta, eta_Delta)
    base_ok = st0 in {"interior", "corner_low", "corner_high"}
    if not base_ok or (not np.isfinite(rho0)):
        n = psi_grid.size
        return {
            "psi_grid": psi_grid,
            "solver_ok_endpoints": False,
            "excluded_bad_share": True,
            "bad_share": 1.0,
            "valid_mask": np.zeros(n, dtype=bool),
            "rho_star": np.full(n, np.nan),
            "rK": np.full(n, np.nan),
            "Gamma_eff": np.full(n, np.nan),
            "g_star": np.full(n, np.nan),
            "W_star": np.full(n, np.nan),
            "Delta": np.full(n, np.nan),
            "lam": np.full(n, np.nan),
            "solv_ok": False,
            "gstar_ok": False,
            "mono_ok": False,
            "deriv_ok": False,
            "omega_base": np.nan,
            "solver_status": np.array(["base_fail"] * n, dtype=object),
        }

    base_drag = lam_func(rho0, c_lam, eta_lam) * delta_func(rho0, c_delta, eta_delta)

    n = psi_grid.size
    rho_star = np.full(n, np.nan)
    solver_status = np.empty(n, dtype=object)
    solver_ok = np.zeros(n, dtype=bool)

    deriv_ok = True
    for i, psi in enumerate(psi_grid):
        rho_i, status = solve_rho_star(psi, rho_bar, a, b, c_lam, eta_lam, c_Delta, eta_Delta)
        solver_status[i] = status
        solver_ok[i] = status in {"interior", "corner_low", "corner_high"}
        rho_star[i] = rho_i if solver_ok[i] else np.nan

        if status == "interior":
            if (not np.isfinite(rho_i)) or (rho_i < DERIV_RHO_MIN):
                deriv_ok = False
            else:
                Bp = B_prime(rho_i, c_lam, eta_lam, c_Delta, eta_Delta, DERIV_RHO_MIN)
                if not np.isfinite(Bp):
                    deriv_ok = False

    solver_ok_endpoints = bool(solver_ok[0] and solver_ok[-1])

    lam = lam_func(rho_star, c_lam, eta_lam)
    Delta = Delta_func(rho_star, c_Delta, eta_Delta)
    drag = lam * delta_func(rho_star, c_delta, eta_delta) - base_drag
    Gamma_eff = Gamma - drag
    rK = r_func(rho_star, r0, a, b) - (1.0 - psi_grid) * lam * Delta
    g_star = np.array([g_star_stationary(psi_grid[i], Gamma_eff[i], kappa_c) for i in range(n)])
    W_star = rK - g_star

    valid = solver_ok.copy()
    valid &= np.isfinite(lam) & np.isfinite(Delta)
    valid &= np.isfinite(rK) & np.isfinite(Gamma_eff)
    valid &= np.isfinite(g_star) & np.isfinite(W_star)

    bad_share = 1.0 - float(np.mean(valid))
    excluded_bad_share = bool(bad_share > cfg["max_bad_share"])

    solv_grid = (Wbar - psi_grid * Delta) > (cfg["eps_solv"] * max(1.0, Wbar))
    g_star_ok = np.isfinite(g_star) & (g_star >= -cfg["tol_g"])

    if np.any(valid) and solver_ok_endpoints and (not excluded_bad_share):
        solv_ok = bool(np.all(solv_grid[valid]))
        gstar_ok = bool(np.all(g_star_ok[valid]))
        mono_ok = bool(
            mono_nondec(rho_star[valid], tol=1e-7)
            and mono_nondec(drag[valid], tol=1e-9)
            and mono_noninc(g_star[valid], tol=1e-9)
            and mono_nondec(W_star[valid], tol=1e-9)
        )
    else:
        solv_ok = False
        gstar_ok = False
        mono_ok = False

    Wabs = np.abs(W_star[valid]) if np.any(valid) else np.array([])
    omega_base = float(np.median(Wabs)) if Wabs.size else np.nan
    if (not np.isfinite(omega_base)) or (omega_base <= 0.0):
        omega_base = np.nan
    else:
        omega_base = max(omega_base, cfg["OMEGA_MIN"])

    return {
        "psi_grid": psi_grid,
        "solver_ok_endpoints": solver_ok_endpoints,
        "excluded_bad_share": excluded_bad_share,
        "bad_share": float(bad_share),
        "valid_mask": valid,
        "rho_star": rho_star,
        "rK": rK,
        "Gamma_eff": Gamma_eff,
        "g_star": g_star,
        "W_star": W_star,
        "Delta": Delta,
        "lam": lam,
        "solv_ok": bool(solv_ok),
        "gstar_ok": bool(gstar_ok),
        "mono_ok": bool(mono_ok),
        "deriv_ok": bool(deriv_ok),
        "omega_base": omega_base,
        "solver_status": solver_status,
    }


def make_grid_lookups(objs, theta):
    """
    Returns lookup functions using linear interpolation on the valid grid support.
    """
    psi_grid = objs["psi_grid"]
    valid = objs["valid_mask"]

    if not (valid[0] and valid[-1]):
        raise ValueError("Grid lookup called without endpoint coverage.")

    x = psi_grid[valid]
    rK_arr = objs["rK"][valid]
    Ge_arr = objs["Gamma_eff"][valid]
    kappa_c = theta["kappa_c"]

    def rK_of_psi(psi):
        psi_clipped = float(np.clip(psi, x[0], x[-1]))
        return float(np.interp(psi_clipped, x, rK_arr))

    def Ge_of_psi(psi):
        psi_clipped = float(np.clip(psi, x[0], x[-1]))
        return float(np.interp(psi_clipped, x, Ge_arr))

    def gstar_of_psi(psi):
        return float(g_star_stationary(psi, Ge_of_psi(psi), kappa_c))

    return rK_of_psi, Ge_of_psi, gstar_of_psi


# =============================================================================
# Exact positivity-preserving Riccati step for g
# =============================================================================
def riccati_g_step(g, Ge, a_coeff, chi, dt):
    """
    Exact solution of  dg/dt = chi * (Ge - g - a * g^2)  over interval dt,
    starting from g >= 0.
    """
    if Ge <= 0.0:
        return g * np.exp(-chi * dt)

    if a_coeff < 1e-14:
        return Ge + (g - Ge) * np.exp(-chi * dt)

    disc = 1.0 + 4.0 * a_coeff * Ge

    if disc < 0.0:
        return g * np.exp(-chi * dt)

    sqrt_disc = np.sqrt(disc)
    g1 = (-1.0 + sqrt_disc) / (2.0 * a_coeff)
    g2 = (-1.0 - sqrt_disc) / (2.0 * a_coeff)

    sep = g1 - g2
    exp_factor = np.exp(-chi * a_coeff * sep * dt)

    denom_0 = g - g2
    if abs(denom_0) < 1e-30:
        return g1

    R0 = (g - g1) / denom_0
    R_new = R0 * exp_factor

    denom = 1.0 - R_new
    if abs(denom) < 1e-30:
        return g1

    g_new = (g1 - R_new * g2) / denom
    return g_new


# =============================================================================
# Break diagnostics on g
# =============================================================================
def compute_break_diagnostics_fixed(g_series, t_series):
    """
    Detects structural breaks and collapse episodes in the growth path.

    Collapse threshold: robust (median - 1.5*IQR of early path, floored at 0).
    Collapse defined as 5 consecutive observations below threshold.
    """
    if len(g_series) < 20:
        return {
            "break_score_mean": np.nan,
            "break_score_var": np.nan,
            "t_break_hat": np.nan,
            "collapse_flag": False,
            "t_collapse_hat": np.nan,
        }

    g_arr = np.array(g_series)
    t_arr = np.array(t_series)
    n = len(g_arr)
    min_segment = max(5, n // 10)

    # Break detection
    breaks_mean = []
    breaks_var = []
    break_times = []

    for split_idx in range(min_segment, n - min_segment):
        g_left = g_arr[:split_idx]
        g_right = g_arr[split_idx:]

        delta_mean = abs(np.mean(g_left) - np.mean(g_right))
        delta_var = abs(np.var(g_left) - np.var(g_right))

        breaks_mean.append(delta_mean)
        breaks_var.append(delta_var)
        break_times.append(t_arr[split_idx])

    if not breaks_mean:
        break_score_mean = np.nan
        break_score_var = np.nan
        t_break_hat = np.nan
    else:
        max_idx = int(np.argmax(breaks_mean))
        break_score_mean = float(breaks_mean[max_idx])
        break_score_var = float(breaks_var[max_idx])
        t_break_hat = float(break_times[max_idx])

    # Robust collapse threshold: median - 1.5*IQR of early path, floored at 0
    first_quarter = g_arr[:len(g_arr)//4]
    if len(first_quarter) > 0:
        q25, q75 = np.percentile(first_quarter, [25, 75])
        iqr = q75 - q25
        median = np.median(first_quarter)
        g_threshold = median - 1.5 * iqr
        g_threshold = max(g_threshold, 0.0)
    else:
        g_threshold = 0.0

    below_threshold = g_arr < g_threshold

    collapse_flag = False
    t_collapse_hat = np.nan

    for i in range(len(below_threshold) - 5 + 1):
        if np.all(below_threshold[i:i+5]):
            collapse_flag = True
            t_collapse_hat = float(t_arr[i])
            break

    return {
        "break_score_mean": float(break_score_mean) if np.isfinite(break_score_mean) else np.nan,
        "break_score_var": float(break_score_var) if np.isfinite(break_score_var) else np.nan,
        "t_break_hat": float(t_break_hat) if np.isfinite(t_break_hat) else np.nan,
        "collapse_flag": bool(collapse_flag),
        "t_collapse_hat": float(t_collapse_hat) if np.isfinite(t_collapse_hat) else np.nan,
    }


# =============================================================================
# Dynamics + simulation
# =============================================================================
def phi_tanh(W, omega):
    return np.tanh(W / omega)


def project_psi(psiP, psi_L):
    """Project psiP into [0, 1 - psi_L]. Returns (projected, clipped_bool)."""
    psiP_proj = np.clip(psiP, 0.0, 1.0 - psi_L)
    return psiP_proj, (psiP_proj != psiP)


def rhs_psi(psiP, g, psi_L, rK_of_psi, omega, sigma):
    """RHS for psi equation only (g treated as parameter)."""
    psi = psi_L + psiP
    W = rK_of_psi(psi) - g
    return sigma * phi_tanh(W, omega)


def rhs_1d(psiP, psi_L, rK_of_psi, gstar_of_psi, dyn):
    psi = psi_L + psiP
    Wstar = rK_of_psi(psi) - gstar_of_psi(psi)
    return dyn["sigma"] * phi_tanh(Wstar, dyn["omega"])


def simulate_2d(psi0, g0, theta, dyn, sim, rK_of_psi, Ge_of_psi, gstar_of_psi, n_steps):
    """
    2D simulation with online fixed-point detection, amplitude-gated cycle
    classification, and sparse break diagnostics on g.

    Attractor types: low / high / interior_fixed / cycle_like / wandering
    """
    psi_L = theta["psi_L"]
    kappa_c = theta["kappa_c"]
    chi = dyn["chi"]
    sigma = dyn["sigma"]
    omega = dyn["omega"]
    dt = sim["dt"]
    eps_psi, eps_g, stable_window = sim["eps_psi"], sim["eps_g"], sim["stable_window"]

    psiP = np.clip(psi0 - psi_L, 0.0, 1.0 - psi_L)
    g = max(0.0, g0)

    max_history_len = max(stable_window * 2, 500)
    psi_history = deque(maxlen=max_history_len)
    g_history = deque(maxlen=max_history_len)

    # Sparse storage for break diagnostics
    save_every = 10
    g_series_sparse = []
    t_series_sparse = []

    clip_psi_total = 0
    clip_g_total = 0
    sup_gap_scaled = 0.0

    # Online fixed-point detection
    attractor_type = "wandering"
    psi_star_hat = np.nan
    g_star_hat = np.nan
    settle_step = np.nan

    for k in range(1, n_steps + 1):
        # Half-step g before RK4 for psi
        psi_now = psi_L + psiP
        psi_now = np.clip(psi_now, psi_L, 1.0)
        Ge_now = Ge_of_psi(psi_now)
        a_now = kappa_c * psi_now
        g_half = riccati_g_step(g, Ge_now, a_now, chi, 0.5 * dt)

        if not np.isfinite(g_half) or g_half < 0.0:
            g_half = max(0.0, g)

        # RK4 for psi
        k1 = rhs_psi(psiP, g, psi_L, rK_of_psi, omega, sigma)
        k2 = rhs_psi(psiP + 0.5 * dt * k1, g_half, psi_L, rK_of_psi, omega, sigma)
        k3 = rhs_psi(psiP + 0.5 * dt * k2, g_half, psi_L, rK_of_psi, omega, sigma)
        k4 = rhs_psi(psiP + dt * k3, g_half, psi_L, rK_of_psi, omega, sigma)

        psiP_new = psiP + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

        # Riccati step for g at midpoint psi
        psi_mid = psi_L + 0.5 * (psiP + psiP_new)
        psi_mid = np.clip(psi_mid, psi_L, 1.0)
        Ge_mid = Ge_of_psi(psi_mid)
        a_coeff = kappa_c * psi_mid

        g_new = riccati_g_step(g, Ge_mid, a_coeff, chi, dt)

        # Project
        psiP_new, hit_psi = project_psi(psiP_new, psi_L)
        if hit_psi:
            clip_psi_total += 1

        if not np.isfinite(g_new):
            g_new = 0.0
            clip_g_total += 1
        elif g_new < 0.0:
            g_new = 0.0
            clip_g_total += 1

        # Gap
        psi_current = psi_L + psiP_new
        gstar_current = gstar_of_psi(psi_current)
        gap_scaled = abs(g_new - gstar_current) / (1.0 + abs(g_new))
        sup_gap_scaled = max(sup_gap_scaled, gap_scaled)

        psiP = psiP_new
        g = g_new

        psi_history.append(psi_L + psiP)
        g_history.append(g)

        # Sparse save for break diagnostics
        if k % save_every == 0:
            g_series_sparse.append(g)
            t_series_sparse.append(k * dt)

        # Online fixed-point detection
        if len(psi_history) >= stable_window:
            psi_tail = list(psi_history)[-stable_window:]
            g_tail = list(g_history)[-stable_window:]

            psi_range = max(psi_tail) - min(psi_tail)
            g_range = max(g_tail) - min(g_tail)

            if psi_range < eps_psi and g_range < eps_g:
                psi_mean = float(np.mean(psi_tail))
                g_mean = float(np.mean(g_tail))
                settle_step = float(k)

                if abs(psi_mean - psi_L) < eps_psi:
                    attractor_type = "low"
                elif abs(psi_mean - 1.0) < eps_psi:
                    attractor_type = "high"
                else:
                    attractor_type = "interior_fixed"

                psi_star_hat = psi_mean
                g_star_hat = g_mean
                break  # Early stop

    # Record observation time
    t_obs = k * dt if k <= n_steps else n_steps * dt

    # If no fixed point detected, classify from final tail
    if attractor_type == "wandering" and len(psi_history) >= stable_window:
        psi_tail = list(psi_history)[-stable_window:]
        g_tail = list(g_history)[-stable_window:]

        psi_range = max(psi_tail) - min(psi_tail)
        g_range = max(g_tail) - min(g_tail)

        # Cycle detection requires substantive amplitude:
        # psi range > 1% of state space, g range > 0.1% of median
        psi_range_min = 0.01 * (1.0 - psi_L)
        g_median = np.median(g_tail)
        g_range_min = 0.001 * max(1.0, abs(g_median))

        if psi_range > psi_range_min and g_range > g_range_min:
            dpsi = np.diff(psi_tail)
            dg = np.diff(g_tail)

            sign_changes_psi = np.sum(dpsi[:-1] * dpsi[1:] < 0)
            sign_changes_g = np.sum(dg[:-1] * dg[1:] < 0)

            if sign_changes_psi > stable_window * 0.3 and sign_changes_g > stable_window * 0.3:
                attractor_type = "cycle_like"

    t_settle = settle_step * dt if np.isfinite(settle_step) else np.nan

    break_diag = compute_break_diagnostics_fixed(g_series_sparse, t_series_sparse)

    return {
        # PRIMARY OUTPUTS
        "attractor_type": attractor_type,
        "psi_star_hat": float(psi_star_hat) if np.isfinite(psi_star_hat) else np.nan,
        "g_star_hat": float(g_star_hat) if np.isfinite(g_star_hat) else np.nan,
        "t_settle": float(t_settle) if np.isfinite(t_settle) else np.nan,
        "t_obs": float(t_obs),
        **break_diag,

        # CROSS-CHECKS / NUMERICS
        "clip_psi": int(clip_psi_total),
        "clip_g": int(clip_g_total),
        "sup_gap_scaled": float(sup_gap_scaled),

        # BACKWARD COMPAT (deprecated)
        "converged": bool(attractor_type in ["low", "high", "interior_fixed"]),
        "t_conv": float(t_settle) if np.isfinite(t_settle) else np.nan,
        "terminal_basin": attractor_type if attractor_type in ["low", "high"] else "none",
    }


def simulate_1d(psi0, theta, dyn, sim, rK_of_psi, gstar_of_psi, n_steps):
    """1D quasi-static simulation."""
    psi_L = theta["psi_L"]
    dt = sim["dt"]
    eps_psi, eps_g, stable_window = sim["eps_psi"], sim["eps_g"], sim["stable_window"]

    gL = gstar_of_psi(psi_L)
    gH = gstar_of_psi(1.0)

    psiP = np.clip(psi0 - psi_L, 0.0, 1.0 - psi_L)

    clip_psi_total = 0
    stable_count = 0
    t_conv = np.nan
    basin = "none"

    for k in range(1, n_steps + 1):
        k1 = rhs_1d(psiP, psi_L, rK_of_psi, gstar_of_psi, dyn)
        k2 = rhs_1d(psiP + 0.5 * dt * k1, psi_L, rK_of_psi, gstar_of_psi, dyn)
        k3 = rhs_1d(psiP + 0.5 * dt * k2, psi_L, rK_of_psi, gstar_of_psi, dyn)
        k4 = rhs_1d(psiP + dt * k3, psi_L, rK_of_psi, gstar_of_psi, dyn)

        psiP_new = psiP + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

        psiP_new, hit_psi = project_psi(psiP_new, psi_L)
        if hit_psi:
            clip_psi_total += 1

        psiP = psiP_new

        psi = psi_L + psiP
        g = gstar_of_psi(psi)

        if abs(psi - psi_L) <= eps_psi and abs(g - gL) <= eps_g:
            b = "low"
        elif abs(psi - 1.0) <= eps_psi and abs(g - gH) <= eps_g:
            b = "high"
        else:
            b = "none"

        if b != "none":
            stable_count += 1
            if stable_count >= stable_window and np.isnan(t_conv):
                t_conv = k * dt
                basin = b
                break
        else:
            stable_count = 0

    return {
        "clip_psi": int(clip_psi_total),
        "converged": bool(basin != "none"),
        "t_conv": float(t_conv) if np.isfinite(t_conv) else np.nan,
        "terminal_basin": basin,
    }


# =============================================================================
# Blocks A/B (IC construction)
# =============================================================================
def ic_list_block_A(psi0, gstar0):
    s = g_scale(gstar0)
    eps = IC_NEAR_EPS * s
    return [
        ("A_on", float(gstar0)),
        ("A_below", float(max(0.0, gstar0 - eps))),
        ("A_above", float(gstar0 + eps)),
    ]


def ic_list_block_B(psi0, gstar0):
    """Cap moderate_above IC to scale-stable displacement."""
    s = g_scale(gstar0)

    g0_below = float(max(0.0, gstar0 - IC_FAR_BELOW_MULT * s))

    bump = IC_FAR_ABOVE_MULT * s
    cap = IC_FAR_ABOVE_CAP_MULT * max(1e-12, abs(gstar0))
    bump_eff = min(bump, cap)
    g0_above = float(gstar0 + bump_eff)

    return [
        ("B_moderate_below", g0_below),
        ("B_moderate_above", g0_above),
    ]


# =============================================================================
# Country-block aggregation
# =============================================================================
def aggregate_country_block(traj_df):
    """
    Aggregates trajectory-level outputs to country-block level.
    break_top_quartile: global rank flag (top 25% of break intensity).
    misclass_boundary_only: boundary basin mismatch between 2D and 1D attractors.
    """
    if traj_df.empty:
        return pd.DataFrame()

    grp = traj_df.groupby(["country_idx", "block"], observed=False)

    def _agg(g):
        share_low = float(np.mean(g["attractor_2D"] == "low"))
        share_high = float(np.mean(g["attractor_2D"] == "high"))
        share_interior_fixed = float(np.mean(g["attractor_2D"] == "interior_fixed"))
        share_cycle_like = float(np.mean(g["attractor_2D"] == "cycle_like"))
        share_wandering = float(np.mean(g["attractor_2D"] == "wandering"))

        break_top_quartile_rate = float(np.mean(g["break_top_quartile"])) if "break_top_quartile" in g.columns else np.nan
        collapse_rate = float(np.mean(g["collapse_flag"]))

        # t_obs statistics
        t_obs_mean = float(np.nanmean(g["t_obs"])) if "t_obs" in g.columns else np.nan
        t_obs_median = float(np.nanmedian(g["t_obs"])) if "t_obs" in g.columns else np.nan

        return pd.Series({
            "N_traj": len(g),
            # PRIMARY OUTPUTS
            "share_low": share_low,
            "share_high": share_high,
            "share_interior_fixed": share_interior_fixed,
            "share_cycle_like": share_cycle_like,
            "share_wandering": share_wandering,
            "break_top_quartile_rate": break_top_quartile_rate,
            "collapse_rate": collapse_rate,
            "t_obs_mean": t_obs_mean,
            "t_obs_median": t_obs_median,
            # CROSS-CHECKS (demoted)
            "misclass_boundary_only": float(np.mean(g["misclass"])),
            "share_none_2D_legacy": float(np.mean(g["basin_2D"] == "none")),
            "share_none_1D": float(np.mean(g["basin_1D"] == "none")),
            "tconv_2D_mean": float(np.nanmean(g.loc[g["conv_2D"], "t_conv_2D"])) if np.any(g["conv_2D"]) else np.nan,
            "tconv_1D_mean": float(np.nanmean(g.loc[g["conv_1D"], "t_conv_1D"])) if np.any(g["conv_1D"]) else np.nan,
            "clip_psi_2D_median": float(np.nanmedian(g["clip_psi_2D"])),
            "clip_g_2D_median": float(np.nanmedian(g["clip_g_2D"])),
            "clip_psi_1D_median": float(np.nanmedian(g["clip_psi_1D"])),
            "sup_gap_scaled_median": float(np.nanmedian(g["sup_gap_scaled"])),
            "sup_gap_scaled_p90": float(np.nanpercentile(g["sup_gap_scaled"], 90)),
        })

    out = grp.apply(_agg).reset_index()
    return out


def summarize_bins(country_block_df, bins_log10, block_name):
    d = country_block_df[country_block_df["block"] == block_name].copy()
    if d.empty:
        return pd.DataFrame()

    d["ratio_bin"] = pd.cut(d["log10_ratio"], bins=bins_log10, include_lowest=True)
    grp = d.groupby("ratio_bin", observed=False)

    out = grp.agg(
        N=("country_idx", "count"),
        # PRIMARY METRICS
        share_interior_fixed_median=("share_interior_fixed", "median"),
        share_cycle_like_median=("share_cycle_like", "median"),
        collapse_rate_median=("collapse_rate", "median"),
        break_top_quartile_rate_median=("break_top_quartile_rate", "median"),
        # LEGACY (for comparison)
        misclass_boundary_median=("misclass_boundary_only", "median"),
        share_high_2D_median=("share_high", "median"),
        share_none_2D_median=("share_none_2D_legacy", "median"),
    ).reset_index()

    out["bin_center"] = out["ratio_bin"].apply(
        lambda x: 0.5 * (x.left + x.right) if pd.notna(x) else np.nan
    )
    out["block"] = block_name
    return out


# =============================================================================
# MAIN MC
# =============================================================================
def draw_dyn_params(rng, cfg):
    chi_lo, chi_hi = cfg["chi_range"]
    sig_lo, sig_hi = cfg["sigma_range"]
    om_lo, om_hi = cfg["omega_mult_range"]

    chi = 10.0 ** rng.uniform(np.log10(chi_lo), np.log10(chi_hi))
    sigma = 10.0 ** rng.uniform(np.log10(sig_lo), np.log10(sig_hi))
    omega_mult = rng.uniform(om_lo, om_hi)
    return {"chi": float(chi), "sigma": float(sigma), "omega_mult": float(omega_mult)}


rng_theta = np.random.default_rng(SEED)
rng_dyn = np.random.default_rng(SEED + 10)

countries = [draw_country_theta(rng_theta, MC_CONFIG) for _ in range(MC_CONFIG["N_COUNTRIES"])]
dyn_list = [draw_dyn_params(rng_dyn, MC_CONFIG) for _ in range(MC_CONFIG["N_COUNTRIES"])]

audit_rows = []
traj_rows = []

for i, theta in enumerate(countries):
    objs = build_manifold_objects(theta, MC_CONFIG)

    solver_ok_endpoints = objs["solver_ok_endpoints"]
    excluded_bad_share = objs["excluded_bad_share"]
    omega_base = objs["omega_base"]

    dyn = dict(dyn_list[i])
    dyn["kappa_c"] = theta["kappa_c"]

    ratio = dyn["chi"] / dyn["sigma"]
    log10_ratio = float(np.log10(ratio))

    omega_ok = np.isfinite(omega_base) and (omega_base > 0.0)
    admissible = bool(solver_ok_endpoints and (not excluded_bad_share) and omega_ok)

    audit_rows.append({
        "country_idx": i,
        "solver_ok_endpoints": bool(solver_ok_endpoints),
        "excluded_bad_share": bool(excluded_bad_share),
        "bad_share": float(objs["bad_share"]),
        "omega_ok": bool(omega_ok),
        "omega_base": float(omega_base) if np.isfinite(omega_base) else np.nan,
        "deriv_ok": bool(objs["deriv_ok"]),
        "solv_ok": bool(objs["solv_ok"]),
        "gstar_ok": bool(objs["gstar_ok"]),
        "mono_ok": bool(objs["mono_ok"]),
        "admissible": admissible,
        "chi": dyn["chi"], "sigma": dyn["sigma"], "omega_mult": dyn["omega_mult"],
        "ratio": ratio, "log10_ratio": log10_ratio,
        "psi_L": theta["psi_L"],
        "Wbar": theta["Wbar"],
        "Gamma": theta["Gamma"],
        "kappa_c": theta["kappa_c"],
        "rho_bar": theta["rho_bar"],
        "a": theta["a"],
        "b": theta["b"],
        "r0": theta["r0"],
        "c_lambda": theta["c_lambda"],
        "eta_lambda": theta["eta_lambda"],
        "c_Delta": theta["c_Delta"],
        "eta_Delta": theta["eta_Delta"],
        "c_delta": theta["c_delta"],
        "eta_delta": theta["eta_delta"],
    })

    if not admissible:
        continue

    dyn["omega"] = float(max(MC_CONFIG["OMEGA_MIN"], dyn["omega_mult"] * omega_base))

    rK_of_psi, Ge_of_psi, gstar_of_psi = make_grid_lookups(objs, theta)

    s_val = max(dyn["sigma"], 1e-14)
    T_i = SIM_CONFIG["T_base"] * max(1.0, SIM_CONFIG["SIGMA_SCALE_REF"] / s_val)
    T_i = min(T_i, SIM_CONFIG["T_cap"])
    n_steps_i = int(np.ceil(T_i / SIM_CONFIG["dt"]))

    psi0_grid = np.linspace(theta["psi_L"], 1.0, MC_CONFIG["psi0_grid_n"])

    for psi0 in psi0_grid:
        gstar0 = gstar_of_psi(psi0)

        # Block A
        for ic_type, g0 in ic_list_block_A(psi0, gstar0):
            out2 = simulate_2d(psi0, g0, theta, dyn, SIM_CONFIG,
                               rK_of_psi, Ge_of_psi, gstar_of_psi, n_steps_i)
            out1 = simulate_1d(psi0, theta, dyn, SIM_CONFIG,
                               rK_of_psi, gstar_of_psi, n_steps_i)

            misclass = int(out2["terminal_basin"] != out1["terminal_basin"])

            traj_rows.append({
                "country_idx": i,
                "block": "A",
                "ic_type": ic_type,
                "psi0": float(psi0),
                "g0": float(g0),
                "chi": dyn["chi"], "sigma": dyn["sigma"], "omega": dyn["omega"],
                "ratio": ratio, "log10_ratio": log10_ratio,
                # PRIMARY OUTPUTS
                "attractor_2D": out2["attractor_type"],
                "psi_star_2D": out2["psi_star_hat"],
                "g_star_2D": out2["g_star_hat"],
                "t_settle_2D": out2["t_settle"],
                "t_obs": out2["t_obs"],
                "break_score_mean": out2["break_score_mean"],
                "break_score_var": out2["break_score_var"],
                "t_break_hat": out2["t_break_hat"],
                "collapse_flag": out2["collapse_flag"],
                "t_collapse_hat": out2["t_collapse_hat"],
                # 1D
                "basin_1D": out1["terminal_basin"],
                "conv_1D": out1["converged"],
                "t_conv_1D": out1["t_conv"],
                "clip_psi_1D": out1["clip_psi"],
                # CROSS-CHECKS (demoted)
                "basin_2D": out2["terminal_basin"],
                "conv_2D": out2["converged"],
                "t_conv_2D": out2["t_conv"],
                "misclass": misclass,
                "clip_psi_2D": out2["clip_psi"],
                "clip_g_2D": out2["clip_g"],
                "T_horizon": T_i,
                "sup_gap_scaled": out2["sup_gap_scaled"],
                "sup_psi_diff": np.nan,
            })

        # Block B
        for ic_type, g0 in ic_list_block_B(psi0, gstar0):
            out2 = simulate_2d(psi0, g0, theta, dyn, SIM_CONFIG,
                               rK_of_psi, Ge_of_psi, gstar_of_psi, n_steps_i)
            out1 = simulate_1d(psi0, theta, dyn, SIM_CONFIG,
                               rK_of_psi, gstar_of_psi, n_steps_i)

            misclass = int(out2["terminal_basin"] != out1["terminal_basin"])

            traj_rows.append({
                "country_idx": i,
                "block": "B",
                "ic_type": ic_type,
                "psi0": float(psi0),
                "g0": float(g0),
                "chi": dyn["chi"], "sigma": dyn["sigma"], "omega": dyn["omega"],
                "ratio": ratio, "log10_ratio": log10_ratio,
                # PRIMARY OUTPUTS
                "attractor_2D": out2["attractor_type"],
                "psi_star_2D": out2["psi_star_hat"],
                "g_star_2D": out2["g_star_hat"],
                "t_settle_2D": out2["t_settle"],
                "t_obs": out2["t_obs"],
                "break_score_mean": out2["break_score_mean"],
                "break_score_var": out2["break_score_var"],
                "t_break_hat": out2["t_break_hat"],
                "collapse_flag": out2["collapse_flag"],
                "t_collapse_hat": out2["t_collapse_hat"],
                # 1D
                "basin_1D": out1["terminal_basin"],
                "conv_1D": out1["converged"],
                "t_conv_1D": out1["t_conv"],
                "clip_psi_1D": out1["clip_psi"],
                # CROSS-CHECKS (demoted)
                "basin_2D": out2["terminal_basin"],
                "conv_2D": out2["converged"],
                "t_conv_2D": out2["t_conv"],
                "misclass": misclass,
                "clip_psi_2D": out2["clip_psi"],
                "clip_g_2D": out2["clip_g"],
                "T_horizon": T_i,
                "sup_gap_scaled": out2["sup_gap_scaled"],
                "sup_psi_diff": np.nan,
            })

# Assemble dataframes
df_audit = pd.DataFrame(audit_rows)
df_traj = pd.DataFrame(traj_rows)

# Compute global break_top_quartile (top 25% of break intensity across all trajectories)
if not df_traj.empty and "break_score_mean" in df_traj.columns:
    global_break_cutoff = df_traj["break_score_mean"].quantile(0.75)
    df_traj["break_top_quartile"] = df_traj["break_score_mean"] > global_break_cutoff
    print(f"\nGlobal break cutoff (p75) = {global_break_cutoff:.6f}")
    print(f"  (break_top_quartile: top 25% of break intensity, not an absolute event flag)")
else:
    df_traj["break_top_quartile"] = False

df_country_block = aggregate_country_block(df_traj)

if not df_country_block.empty:
    ratio_map = df_audit.loc[
        df_audit["admissible"], ["country_idx", "ratio", "log10_ratio"]
    ].drop_duplicates()
    df_country_block = df_country_block.merge(ratio_map, on="country_idx", how="left")

df_bins_A = summarize_bins(df_country_block, MC_CONFIG["ratio_bins_log10"], "A")
df_bins_B = summarize_bins(df_country_block, MC_CONFIG["ratio_bins_log10"], "B")

# Save outputs
paths = {}
paths["audit"] = save_df(df_audit, "audit")
paths["traj"] = save_df(df_traj, "traj")
paths["country_block"] = save_df(df_country_block, "country_block")
paths["bins_A"] = save_df(df_bins_A, "bins_blockA")
paths["bins_B"] = save_df(df_bins_B, "bins_blockB")


# Plots
def plot_block(block_name, df_cb, df_bins):
    d = df_cb[df_cb["block"] == block_name].copy()
    if d.empty or df_bins.empty:
        return {}
    out = {}

    # PRIMARY METRICS
    out["interior_fixed"] = save_plot_scatter_with_binned(
        d, "log10_ratio", "share_interior_fixed", df_bins, "share_interior_fixed_median",
        f"Interior Fixed Attractors vs log10(chi/sigma) — Block {block_name}",
        f"interior_fixed_vs_ratio_block{block_name}",
    )
    out["collapse"] = save_plot_scatter_with_binned(
        d, "log10_ratio", "collapse_rate", df_bins, "collapse_rate_median",
        f"Collapse Rate vs log10(chi/sigma) — Block {block_name}",
        f"collapse_vs_ratio_block{block_name}",
    )
    out["break_top_quartile"] = save_plot_scatter_with_binned(
        d, "log10_ratio", "break_top_quartile_rate", df_bins, "break_top_quartile_rate_median",
        f"Break Top Quartile Rate vs log10(chi/sigma) — Block {block_name}",
        f"break_topq_vs_ratio_block{block_name}",
    )
    # LEGACY (for comparison)
    out["misclass_boundary"] = save_plot_scatter_with_binned(
        d, "log10_ratio", "misclass_boundary_only", df_bins, "misclass_boundary_median",
        f"Boundary Basin Mismatch vs log10(chi/sigma) — Block {block_name}",
        f"misclass_boundary_vs_ratio_block{block_name}",
    )
    return out


plots = {}
plots["A"] = plot_block("A", df_country_block, df_bins_A)
plots["B"] = plot_block("B", df_country_block, df_bins_B)

# Config dump
config_dump = {
    "MC_NAME": MC_NAME,
    "RUN_TS": RUN_TS,
    "SEED": SEED,
    "MC_CONFIG": MC_CONFIG,
    "SIM_CONFIG": SIM_CONFIG,
    "IC_NEAR_EPS": IC_NEAR_EPS,
    "IC_FAR_BELOW_MULT": IC_FAR_BELOW_MULT,
    "IC_FAR_ABOVE_MULT": IC_FAR_ABOVE_MULT,
    "IC_FAR_ABOVE_CAP_MULT": IC_FAR_ABOVE_CAP_MULT,
    "platform": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": plt.matplotlib.__version__,
    },
}
config_path = os.path.join(OUT_DIR, "config_dump.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_dump, f, indent=2)
paths["config_dump"] = config_path

# Console summary
N_total = int(MC_CONFIG["N_COUNTRIES"])
N_adm = int(df_audit["admissible"].sum()) if not df_audit.empty else 0
print("=" * 100)
print(f"mc_block_2 — Dynamic Stability MC complete | MC_NAME={MC_NAME} | RUN_TS={RUN_TS}")
print("=" * 100)
print(f"N_total={N_total} | admissible={N_adm}")
print()

if not df_audit.empty:
    print("Audit (hard filters):")
    print(df_audit[["solver_ok_endpoints", "excluded_bad_share", "omega_ok", "admissible"]].mean().to_string())
    print()

    prim_cols = ["psi_L", "Wbar", "Gamma", "kappa_c", "rho_bar", "a", "b", "r0",
                 "c_lambda", "eta_lambda", "c_Delta", "eta_Delta", "c_delta", "eta_delta"]
    missing = [c for c in prim_cols if c not in df_audit.columns]
    if missing:
        print(f"WARNING: audit missing primitive columns: {missing}")
    else:
        print(f"Audit replayable: all {len(prim_cols)} primitive columns present.")
    print()

print("Saved outputs:")
for k, v in paths.items():
    print(f"  {k:18s} -> {v}")

if not df_traj.empty:
    print(f"\nTrajectory count: {len(df_traj)}")

    settled = df_traj[df_traj["attractor_2D"].isin(["low", "high", "interior_fixed"])]
    if not settled.empty:
        print(f"\nt_settle for settled trajectories:")
        print(f"  min={settled['t_settle_2D'].min():.1f}  "
              f"median={settled['t_settle_2D'].median():.1f}  "
              f"max={settled['t_settle_2D'].max():.1f}")

    print(f"\nt_obs (observation time) variation:")
    print(f"  min={df_traj['t_obs'].min():.1f}  "
          f"median={df_traj['t_obs'].median():.1f}  "
          f"max={df_traj['t_obs'].max():.1f}")

    print(f"\nT_horizon variation:")
    print(f"  min={df_traj['T_horizon'].min():.1f}  "
          f"median={df_traj['T_horizon'].median():.1f}  "
          f"max={df_traj['T_horizon'].max():.1f}")

    print("\n" + "=" * 80)
    print("PRIMARY OUTPUTS: Attractor Classification")
    print("=" * 80)
    for bl in ["A", "B"]:
        sub = df_traj[df_traj["block"] == bl]
        if sub.empty:
            continue
        print(f"\nBlock {bl}:")
        print(f"  low={np.mean(sub['attractor_2D']=='low'):.3f}  "
              f"high={np.mean(sub['attractor_2D']=='high'):.3f}  "
              f"interior_fixed={np.mean(sub['attractor_2D']=='interior_fixed'):.3f}")
        print(f"  cycle_like={np.mean(sub['attractor_2D']=='cycle_like'):.3f}  "
              f"wandering={np.mean(sub['attractor_2D']=='wandering'):.3f}")
        print(f"  collapse_rate={np.mean(sub['collapse_flag']):.3f}  "
              f"break_top_quartile_rate={np.mean(sub['break_top_quartile']):.3f}")
        print(f"  break_score_mean: median={sub['break_score_mean'].median():.4f}  "
              f"p90={sub['break_score_mean'].quantile(0.9):.4f}")

    print("\n" + "=" * 80)
    print("CROSS-CHECKS (demoted, for comparison only)")
    print("=" * 80)
    for bl in ["A", "B"]:
        sub = df_traj[df_traj["block"] == bl]
        if sub.empty:
            continue
        print(f"\nBlock {bl}:")
        print(f"  misclass_boundary_only={sub['misclass'].mean():.6f}  "
              f"sup_gap_scaled_p90={sub['sup_gap_scaled'].quantile(0.9):.6f}")

    print("\nGap summary (sup_gap_scaled):")
    for bl in ["A", "B"]:
        sub = df_traj[df_traj["block"] == bl]
        if sub.empty:
            continue
        print(f"  Block {bl}:")
        print(f"    median={sub['sup_gap_scaled'].median():.6f}  "
              f"p90={sub['sup_gap_scaled'].quantile(0.9):.6f}  "
              f"max={sub['sup_gap_scaled'].max():.6f}")

    print("\nClip summary:")
    for bl in ["A", "B"]:
        sub = df_traj[df_traj["block"] == bl]
        if sub.empty:
            continue
        print(f"  Block {bl}:")
        print(f"    clip_g_2D  — median={sub['clip_g_2D'].median():.0f}  "
              f"p90={sub['clip_g_2D'].quantile(0.9):.0f}  "
              f"max={sub['clip_g_2D'].max():.0f}")
        print(f"    clip_psi_2D — median={sub['clip_psi_2D'].median():.0f}  "
              f"p90={sub['clip_psi_2D'].quantile(0.9):.0f}")
        print(f"    clip_psi_1D — median={sub['clip_psi_1D'].median():.0f}  "
              f"p90={sub['clip_psi_1D'].quantile(0.9):.0f}")

print("=" * 100)